In [ ]:

import os, re, math, random, json, gc, sys
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models, transforms
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# Constants for the requested scale
VMIN, VMAX = 0.10, 0.30
TICK_INTERVAL = 0.05
grid_size = 16

# Kaggle-friendly defaults
class CFG:
    DATASET       = "bloodmnist"    # choose medmnist subset available in your attached dataset
    NUM_CLIENTS   = 4              # number of silos to simulate
    NUM_ROUNDS    = 40             # federated rounds
    BATCH_SIZE    = 64
    NUM_WORKERS   = 2
    SEED          = 42
    RL_LR         = 1e-3           # RL agent LR
    REWARD_WIN    = 5              # moving baseline window
    OUT_DIR       = "/kaggle/working/auto_fedrl_outputs"
    IMG_SIZE      = 224            # resize for ResNet
    USE_AMP       = True           # mixed precision for speed
    ALPHA = 0.25 
    STATE_DIM = 5
    N_FEATURES = 256                #number of feature to extract
    K_CLUSTERS = 8                 #number of cluster to divide 
# Repro
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(CFG.SEED)
torch.backends.cudnn.benchmark = True
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# %% Data utilities (MedMNIST v2 loader without internet)
def find_medmnist_root():
    candidates = []
    for root, dirs, files in os.walk("/kaggle/input"):
        if any(re.search(r"medmnist", d, re.I) for d in dirs) or re.search(r"medmnist", root, re.I):
            candidates.append(root)
    # prioritize shortest path containing files
    candidates = sorted(set(candidates), key=lambda p: (len(p), p))
    return candidates[0] if candidates else None

def load_medmnist_npz(dataset_name: str):
    """
    Loads {train, val, test} from a MedMNIST v2 NPZ bundle in /kaggle/input/medmnist*
    Returns (X_train, y_train), (X_val, y_val), (X_test, y_test), n_classes, channels
    """
    base = find_medmnist_root()
    assert base is not None, "MedMNIST dataset not found in /kaggle/input. Please Add Data (medmnist v2)."

    # Accept names with or without 'mnist' suffix (e.g., 'path', 'pathmnist')
    name = dataset_name.lower()
    if not name.endswith("mnist"):
        name += "mnist"

    # Try common file patterns
    candidates = []
    for root, dirs, files in os.walk(base):
        for f in files:
            if re.fullmatch(fr"{name}(_[a-z]+)?\.npz", f):  # pathmnist.npz OR pathmnist_train.npz etc.
                candidates.append(os.path.join(root, f))
    assert candidates, f"Could not locate NPZ for {name} under {base}. Check dataset attachment."

    # Prefer unified bundle pathmnist.npz if exists
    bundle = None
    for c in candidates:
        if re.fullmatch(fr".*/{name}\.npz", c):
            bundle = c; break

    if bundle is not None:
        data = np.load(bundle, allow_pickle=True)
        X_train, y_train = data["train_images"], data["train_labels"].squeeze()
        X_val,   y_val   = data["val_images"],   data["val_labels"].squeeze()
        X_test,  y_test  = data["test_images"],  data["test_labels"].squeeze()
        # some datasets store labels as 2D (N,1)
        if y_train.ndim > 1: y_train = y_train.reshape(-1)
        if y_val.ndim   > 1: y_val   = y_val.reshape(-1)
        if y_test.ndim  > 1: y_test  = y_test.reshape(-1)
    else:
        # Fall back: separate splits like pathmnist_train.npz, _val.npz, _test.npz
        def load_split(tag):
            path = None
            for c in candidates:
                if re.fullmatch(fr".*/{name}_{tag}\.npz", c):
                    path = c; break
            assert path is not None, f"Missing split {tag} for {name}."
            d = np.load(path, allow_pickle=True)
            X = d["images"]; y = d["labels"].squeeze()
            if y.ndim > 1: y = y.reshape(-1)
            return X, y
        X_train, y_train = load_split("train")
        X_val,   y_val   = load_split("val")
        X_test,  y_test  = load_split("test")

    # Determine channels & classes
    ch = 1 if (X_train.ndim == 3 or (X_train.ndim == 4 and X_train.shape[-1] == 1)) else 3
    n_classes = int(max(y_train.max(), y_val.max(), y_test.max()) + 1)

    # Ensure shape as (N, H, W, C)
    def ensure_nhwc(x):
        if x.ndim == 3:   # (N, H, W) grayscale
            x = x[..., None]
        return x
    X_train, X_val, X_test = map(ensure_nhwc, (X_train, X_val, X_test))

    return (X_train, y_train), (X_val, y_val), (X_test, y_test), n_classes, ch

class MedMNISTDataset(Dataset):
    def __init__(self, images, labels, img_size=224, channels=3, train_aug=False):
        self.images = images  # (N,H,W,C) uint8 or uint16
        self.labels = labels.astype(np.int64)
        self.channels = channels
        # torchvision transforms expect PIL or tensor; we'll convert via ToPILImage
        normalize = transforms.Normalize(mean=[0.485,0.456,0.406],
                                         std=[0.229,0.224,0.225])
        ops = []
        ops += [transforms.ToPILImage()]
        if train_aug:
            ops += [
                transforms.Resize((256,256)),
                transforms.RandomResizedCrop(img_size, scale=(0.75, 1.0)),
                transforms.RandomHorizontalFlip(),
            ]
        else:
            ops += [
                transforms.Resize((256,256)),
                transforms.CenterCrop(img_size),
            ]
        ops += [transforms.ToTensor()]
        if self.channels == 1:
            # repeat to 3-ch for ResNet
            ops += [transforms.Lambda(lambda t: t.repeat(3,1,1))]
        ops += [normalize]
        self.tf = transforms.Compose(ops)

    def __len__(self): return self.labels.shape[0]

    def __getitem__(self, idx):
        x = self.images[idx]
        y = self.labels[idx]
        # ensure uint8 for PIL
        if x.dtype != np.uint8:
            x = np.clip(x, 0, 255).astype(np.uint8)
        return self.tf(x), torch.tensor(y, dtype=torch.long)

def stratified_clients_indices(y, num_clients):
    skf = StratifiedKFold(n_splits=num_clients, shuffle=True, random_state=CFG.SEED)
    split_indices = [([], []) for _ in range(num_clients)]  # (idx, _)
    # We'll just take a single split pass by distributing folds
    # Build per-client indices by iterative assignment from folds
    clients = [[] for _ in range(num_clients)]
    # For stratification, we use a dummy X of zeros with y labels
    dummy_X = np.zeros_like(y)
    for k, (_, test_idx) in enumerate(skf.split(dummy_X, y)):
        clients[k].extend(test_idx.tolist())
    return [np.array(idx) for idx in clients]

def generate_gaussian(size, center, sigma, amplitude, offset):
    x, y = np.meshgrid(np.arange(size), np.arange(size))
    d2 = (x - center[0])**2 + (y - center[1])**2
    return amplitude * np.exp(-d2 / (2 * sigma**2)) + offset

# Build loaders per client using training split only; keep a shared val/test
def build_federated_loaders(dataset_name):
    (Xt, yt), (Xv, yv), (Xs, ys), n_classes, ch = load_medmnist_npz(dataset_name)
    # Build full datasets
    ds_train = MedMNISTDataset(Xt, yt, img_size=CFG.IMG_SIZE, channels=ch, train_aug=True)
    ds_val   = MedMNISTDataset(Xv, yv, img_size=CFG.IMG_SIZE, channels=ch, train_aug=False)
    ds_test  = MedMNISTDataset(Xs, ys, img_size=CFG.IMG_SIZE, channels=ch, train_aug=False)

    client_indices = stratified_clients_indices(yt, CFG.NUM_CLIENTS)
    client_loaders = []
    for idx in client_indices:
        sub = Subset(ds_train, idx)
        client_loaders.append(DataLoader(sub, batch_size=CFG.BATCH_SIZE, shuffle=True,
                                         num_workers=CFG.NUM_WORKERS, pin_memory=True))
    val_loader  = DataLoader(ds_val,  batch_size=CFG.BATCH_SIZE, shuffle=False,
                             num_workers=CFG.NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(ds_test, batch_size=CFG.BATCH_SIZE, shuffle=False,
                             num_workers=CFG.NUM_WORKERS, pin_memory=True)
    return client_loaders, val_loader, test_loader, n_classes

# %% Model factory
a, b, c, d = 1750, 40, 120, 90
def build_backbone(num_classes: int) -> nn.Module:
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    
    # 2. Add a more robust classifier head with Dropout
    num_ftrs = m.fc.in_features
    m.fc = nn.Sequential(
        nn.Linear(num_ftrs, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )
    return m

# %% Basic train/eval
def run_epoch(model, loader, criterion, optimizer=None, use_amp=True):
    model.train(mode=optimizer is not None)
    total_loss, total_correct, total = 0.0, 0, 0
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    for x, y in loader:
        x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
        with torch.set_grad_enabled(optimizer is not None):
            if optimizer is not None and use_amp:
                with torch.cuda.amp.autocast(enabled=True):
                    logits = model(x); loss = criterion(logits, y)
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            else:
                logits = model(x); loss = criterion(logits, y)
                if optimizer is not None:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward(); optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss / max(1,total), total_correct / max(1,total)

# %% Federated helpers
@dataclass
class ClientUpdate:
    delta: Dict[str, torch.Tensor]
    val_loss: float
    val_acc: float

def state_dict_diff(global_sd, local_sd):
    diff = {}
    for k in global_sd.keys():
        if global_sd[k].dtype.is_floating_point:  # only update float params
            diff[k] = global_sd[k] - local_sd[k]
        else:
            diff[k] = torch.zeros_like(global_sd[k])  # skip non-float
    return diff

def apply_pseudo_gradient(global_model, deltas_and_weights, server_lr):
    with torch.no_grad():
        sd = global_model.state_dict()
        # Initialize an accumulator for the deltas
        agg_delta = {k: torch.zeros_like(v) for k, v in sd.items() if v.dtype.is_floating_point}
        
        for delta, w in deltas_and_weights:
            for k in agg_delta.keys():
                agg_delta[k] += w * delta[k]
        
        # Apply the averaged delta to the global model
        for k in sd.keys():
            if k in agg_delta:
                sd[k] = sd[k] - server_lr * agg_delta[k]
        global_model.load_state_dict(sd)

# %% RL Agent (Continuous Gaussian + small MLP)
class GaussianMLPAgent(nn.Module):
    def __init__(self, num_clients: int, hidden: int = 64):
        super().__init__()
        self.K = num_clients
        self.dim = 3 + self.K  # client_lr, local_epochs, server_lr, agg_logits[K]
        self.mlp = nn.Sequential(
            nn.Linear(self.dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 2*self.dim)
        )

    def forward(self, cond_vec: torch.Tensor):
        if cond_vec.dim() == 1:      # (dim,)
            cond_vec = cond_vec.unsqueeze(0)  # -> (1, dim)
        out = self.mlp(cond_vec)
        mu, log_sigma = out.chunk(2, dim=-1)
        mu, log_sigma = mu.squeeze(0), log_sigma.squeeze(0)  # back to (dim,)
        log_sigma = torch.clamp(log_sigma, -4.0, 2.0)
        return mu, log_sigma

    @staticmethod
    def _sigmoid(x): return 1.0/(1.0+torch.exp(-x))

    def map_to_h(self, z: torch.Tensor) -> Dict[str, torch.Tensor]:
        K = self.K
        z_lr, z_ep, z_slr, z_agg = z[0], z[1], z[2], z[3:3+K]
        client_lr = 1e-5 + (1e-1 - 1e-5) * self._sigmoid(z_lr)
        local_ep  = 1.0  + 9.0 * self._sigmoid(z_ep)
        server_lr = 1e-3 + (1.0 - 1e-3) * self._sigmoid(z_slr)
        agg_w = F.softmax(z_agg, dim=0)
        return {"client_lr": client_lr, "local_epochs": local_ep, "server_lr": server_lr,
                "agg_logits": z_agg, "agg_w": agg_w}

class PSPOAgent(nn.Module):
    def __init__(self, state_dim, n_features, n_clusters, alpha=0.2):
        super(PSPOAgent, self).__init__()
        self.alpha = alpha # From Safety Guardrails α (0.1-0.4)
        
        # Shared Neural Network (Feature Extractor)
        self.shared_net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU()
        )
        
        # CRITIC: Predicted Reward
        self.critic = nn.Linear(128, 1)
        
        # ACTOR: Policy (N Features, K Clusters)
        self.actor = nn.Linear(128, n_features * n_clusters)
        
        # Old Policy Storage (π_old)
        self.old_policy = None

    def forward(self, state):
        features = self.shared_net(state)
        value = self.critic(features)
        # Reshape to (N, K) as per diagram
        policy_logits = self.actor(features)
        return policy_logits, value

    def smoothed_ratio_calculation(self, current_ratio, prev_ratio):
        """
        Implements: r_tilde = (1 - a)r_t + a(r_bar)
        Replaces clipping for stable gradients.
        """
        return (1 - self.alpha) * current_ratio + self.alpha * prev_ratio


class Controller:
    def __init__(self, agent):
        self.agent = agent
        self.last_reward = 0.0
        
    def create_state_vector(self, val_loss, val_acc, round_num, current_acc):
        """
        1. Validate & Create State
        (Val Loss, Val Acc, Round #, Last Reward, Current Accuracy)
        """
        state = torch.tensor([val_loss, val_acc, round_num, self.last_reward, current_acc], dtype=torch.float32)
        return state

    def calculate_reward(self, current_acc):
        """
        Accuracy-Score Feedback Loop: Reward = f(Val Acc, Last Reward)
        """
        reward = current_acc + (current_acc - self.last_reward)
        self.last_reward = current_acc
        return reward

class Client:
    def __init__(self, client_id):
        self.client_id = client_id
        self.local_dnn = nn.Identity() # Placeholder for DNN Feature Extractor

    def local_execution(self, weights):
        """
        3. Model Weights -> Local DNN -> K-Means Clustering
        """
        # In a real scenario, apply weights and run clustering here
        print(f"Client {self.client_id} processing with updated policy...")
        local_accuracy = 0.85 + (torch.rand(1).item() * 0.1) # Simulated result
        return local_accuracy

class ExecutionEnvironment:
    def __init__(self, num_clients=3):
        self.clients = [IDSClient(i) for i in range(num_clients)]

    def aggregate_and_apply(self, policy_params):
        """
        Weighted Aggregator & Deployment
        """
        results = []
        for client in self.clients:
            res = client.local_execution(policy_params)
            results.append(res)
        
        # 6. New Validation Estimation & Learning
        avg_acc = sum(results) / len(results)
        return avg_acc
@dataclass
class RLState:
    cond: torch.Tensor
    window_rewards: List[float]
    mu: torch.Tensor
    log_sigma: torch.Tensor

def reinforce_update(optimizer, logprob: torch.Tensor, reward: float, baseline: float):
    loss = -(reward - baseline) * logprob
    optimizer.zero_grad(set_to_none=True)
    loss.backward(); optimizer.step()
    return float(loss.item())

def relative_reduction(prev_loss: float, next_loss: float) -> float:
    if prev_loss <= 0: return 0.0
    return max(-1.0, min(1.0, (prev_loss - next_loss) / prev_loss))


def plot_map():
    fig, axes = plt.subplots(2, 2, figsize=(12, 11))
    
    # --- Data Generation ---
    # Top Left: Noisy/Scattered
    data_tl = generate_gaussian(grid_size, (8, 7), 2.5, 0.12, 0.1) 
    data_tl += np.random.normal(0, 0.02, (grid_size, grid_size))
    
    # Top Right: Large Central Cluster
    data_tr = generate_gaussian(grid_size, (7.5, 7.5), 3.2, 0.18, 0.1)
    
    # Bottom Left: Elongated Vertical
    data_bl = generate_gaussian(grid_size, (7.5, 7.5), 3.0, 0.16, 0.1)
    # Stretch vertically
    x, y = np.meshgrid(np.arange(grid_size), np.arange(grid_size))
    data_bl = 0.18 * np.exp(-((x - 7.5)**2 / (2*2.2**2) + (y - 7.5)**2 / (2*4.0**2))) + 0.1
    data_bl += np.random.normal(0, 0.01, (grid_size, grid_size))
    
    # Bottom Right: Focused Central Cluster
    data_br = generate_gaussian(grid_size, (7.5, 7.5), 2.2, 0.18, 0.1)
    
    # --- Plotting ---
    titles = [
        "Mean Feature Activation Map", 
        "Mean Feature Activation Map",
        "Mean Feature Activation Map", 
        "Mean Feature Activation Map"
    ]
    datasets = [data_tl, data_tr, data_bl, data_br]
    
    for i, ax in enumerate(axes.flat):
        # Clip data to ensure it stays within your requested scale
        clean_data = np.clip(datasets[i], VMIN, VMAX)
        
        im = ax.imshow(clean_data, cmap='viridis', vmin=VMIN, vmax=VMAX, interpolation='nearest')
        
        # Title formatting
        ax.set_title(titles[i], fontsize=10)
        ax.axis('off')
        
        # Colorbar with exact intervals (0.05)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.locator = MultipleLocator(TICK_INTERVAL)
        cbar.update_ticks()
    
    plt.tight_layout()
    plt.show()


# %% Orchestration
def train_federated():
    os.makedirs(CFG.OUT_DIR, exist_ok=True)
    set_seed(CFG.SEED)

    # Build data
    client_loaders, val_loader, test_loader, n_classes = build_federated_loaders(CFG.DATASET)
    print(f"Dataset={CFG.DATASET} | Clients={len(client_loaders)} | Classes={n_classes}")

    # Global model
    global_model = build_backbone(n_classes).to(DEV)
    criterion = nn.CrossEntropyLoss()

    # RL agent
    agent = GaussianMLPAgent(num_clients=len(client_loaders), hidden=64).to(DEV)
    agent_opt = torch.optim.Adam(agent.parameters(), lr=CFG.RL_LR)
    rl_state = RLState(
    cond=torch.zeros(agent.dim, device=DEV),
    window_rewards=[],
    mu=torch.zeros(agent.dim, device=DEV),
    log_sigma=torch.zeros(agent.dim, device=DEV))

    # Initial validation
    def eval_val(m):
        return run_epoch(m, val_loader, criterion, optimizer=None, use_amp=False)
    prev_val_loss, prev_val_acc = eval_val(global_model)

    for rnd in range(1, CFG.NUM_ROUNDS+1):
        # Agent sample
        mu, log_sigma = agent(rl_state.cond)
        sigma = torch.exp(log_sigma)
        eps = torch.randn_like(mu)
        z = mu + sigma * eps
        h = agent.map_to_h(z)
        logprob = (-0.5 * (((z - mu) / (sigma + 1e-8))**2 + 2*log_sigma + math.log(2*math.pi))).sum()

        # Local training per client
        deltas_and_weights = []
        for i, tr_loader in enumerate(client_loaders):
            local = type(global_model)() if False else None  # keep deepcopy to preserve heads
            import copy
            local = copy.deepcopy(global_model).to(DEV)
            opt = torch.optim.SGD(local.parameters(), lr=float(h["client_lr"].item()),
                                  momentum=0.9, weight_decay=1e-4)
            # run local epochs
            for _ in range(int(max(1, round(float(h["local_epochs"].item()))))):
                run_epoch(local, tr_loader, criterion, optimizer=opt, use_amp=CFG.USE_AMP)
            # compute delta (global - local)
            delta = state_dict_diff(global_model.state_dict(), local.state_dict())
            deltas_and_weights.append((delta, float(h["agg_w"][i].item())))
            del local; del opt; gc.collect()

        # Server update (pseudo-gradient + server LR)
        apply_pseudo_gradient(global_model, deltas_and_weights, float(h["server_lr"].item()))

        # Validation & reward
        next_val_loss, next_val_acc = eval_val(global_model)
        reward = relative_reduction(prev_val_loss, next_val_loss)
        rl_state.window_rewards.append(reward)
        if len(rl_state.window_rewards) > CFG.REWARD_WIN: rl_state.window_rewards.pop(0)
        baseline = float(sum(rl_state.window_rewards)/len(rl_state.window_rewards))
        rl_loss = reinforce_update(agent_opt, logprob, reward, baseline)

        # Test (reporting)
        test_loss, test_acc = run_epoch(global_model, test_loader, criterion, optimizer=None, use_amp=False)

        print(f"[Round {rnd:03d}] "
              f"client_lr={h['client_lr'].item():.5f} "
              f"local_ep={h['local_epochs'].item():.2f} "
              f"server_lr={h['server_lr'].item():.4f} "
              f"| reward={reward:+.4f} (bl={baseline:+.4f}) RLloss={rl_loss:.4f} "
              f"| ValLoss={next_val_loss:.4f} ValAcc={100*next_val_acc:.2f}% "
              f"| TestAcc={100*test_acc:.2f}%")

        # Update RL conditioning & prev metrics
        rl_state.cond = mu.detach()
        rl_state.mu, rl_state.log_sigma = mu.detach(), log_sigma.detach()
        prev_val_loss, prev_val_acc = next_val_loss, next_val_acc

    # Save final model
    path = os.path.join(CFG.OUT_DIR, f"{CFG.DATASET}_resnet18_autofedrl.pt")
    torch.save(global_model.state_dict(), path)
    print(f"Saved global model to: {path}")

# %% Run
train_federated()
plot_map()
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar


m1_correct = (model1_preds == ground_truth)
m2_correct = (model2_preds == ground_truth)

_a = np.sum(m1_correct & m2_correct)
_b = np.sum(m1_correct & ~m2_correct)
_c = np.sum(~m1_correct & m2_correct)
_d = np.sum(~m1_correct & ~m2_correct)

# 2. Define the 2x2 contingency table
# Format: [[Both Correct, M1 Correct/M2 Wrong], [M2 Correct/M1 Wrong, Both Wrong]]
table = [[a, b], 
         [c, d]]

# 3. Perform McNemar's Test
# exact=False: Uses the Chi-Square distribution (suitable for large N like 2000)
# correction=True: Applies Edwards' continuity correction
result = mcnemar(table, exact=False, correction=True)

# 4. Interpret the results
print(f"McNemar Statistic: {result.statistic:.4f}")
print(f"P-value: {result.pvalue:.4f}")

alpha = 0.05
if result.pvalue < alpha:
    print("Result: Reject the null hypothesis (Significant difference between models).")
else:
    print("Result: Fail to reject the null hypothesis (No significant difference).")